In [2]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print("1. Loading datasets...")
train_df = pd.read_csv('/kaggle/input/datasets/pavansrikrishnavamsi/test-csv/train.csv')
test_df = pd.read_csv('/kaggle/input/datasets/pavansrikrishnavamsi/test-csv/test.csv')

categorical_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'geohash_hour']

def preprocess_features(df):
    df_clean = df.copy()
    
    # 1. Base Time
    df_clean[['hour', 'minute']] = df_clean['timestamp'].str.split(':', expand=True).astype(float)
    df_clean = df_clean.drop(columns=['timestamp'])
    
    # 2. Cyclical Time
    time_in_mins = (df_clean['hour'] * 60) + df_clean['minute']
    df_clean['sin_time'] = np.sin(2 * np.pi * time_in_mins / 1440)
    df_clean['cos_time'] = np.cos(2 * np.pi * time_in_mins / 1440)
    
    # 3. Real-World Event Flags
    df_clean['is_morning_commute'] = ((df_clean['hour'] >= 7) & (df_clean['hour'] <= 9)).astype(int)
    df_clean['is_school_rush'] = ((df_clean['hour'] >= 15) & (df_clean['hour'] <= 17)).astype(int)
    df_clean['is_dead_night'] = ((df_clean['hour'] >= 23) | (df_clean['hour'] <= 5)).astype(int)
    
    # 4. Spatio-Temporal Interaction
    df_clean['geohash_hour'] = df_clean['geohash'].astype(str) + '_' + df_clean['hour'].astype(str)
    
    # 5. CATBOOST FIX
    for col in categorical_cols:
        df_clean[col] = df_clean[col].fillna('Missing').astype(str).astype('category')
        
    return df_clean

print("2. Engineering proven features...")
train_clean = preprocess_features(train_df)
test_clean = preprocess_features(test_df)

# The Full Dataset
X_FULL = train_clean.drop(columns=['Index', 'demand'])
y_FULL = train_clean['demand']
X_test = test_clean.drop(columns=['Index'])

# The 80/20 Split (Only used to find honest ensemble weights!)
X_train, X_val, y_train, y_val = train_test_split(X_FULL, y_FULL, test_size=0.2, random_state=42)

print("3. Applying Advanced Bayesian Smoothed Target Encoding...")
global_mean = y_FULL.mean()

# The Bayesian Smoothing Function
def get_bayesian_encoding(df, cat_col, target_col, weight=20):
    agg = df.groupby(cat_col)[target_col].agg(['count', 'mean'])
    # Pulls rare occurrences back toward the global mean to stop extreme overfitting
    smoothed = (agg['count'] * agg['mean'] + weight * global_mean) / (agg['count'] + weight)
    return smoothed.to_dict()

# Map the smoothed averages for Phase 1 (The 80/20 Split)
temp_df = pd.concat([X_train, y_train], axis=1)
target_mapper_val = get_bayesian_encoding(temp_df, 'geohash_hour', 'demand', weight=20)
X_train['historical_demand'] = X_train['geohash_hour'].map(target_mapper_val).fillna(global_mean)
X_val['historical_demand'] = X_val['geohash_hour'].map(target_mapper_val).fillna(global_mean)

# Map the smoothed averages for Phase 2 & 3 (The 100% Final Retrain)
full_df = pd.concat([X_FULL, y_FULL], axis=1)
target_mapper_full = get_bayesian_encoding(full_df, 'geohash_hour', 'demand', weight=20)
X_FULL['historical_demand'] = X_FULL['geohash_hour'].map(target_mapper_full).fillna(global_mean)
X_test['historical_demand'] = X_test['geohash_hour'].map(target_mapper_full).fillna(global_mean)


print("\n--- PHASE 1: FINDING HONEST WEIGHTS (80/20 SPLIT) ---")
lgb_temp = LGBMRegressor(n_estimators=1000, learning_rate=0.03, max_depth=8, num_leaves=63, random_state=42, n_jobs=-1)
xgb_temp = XGBRegressor(n_estimators=800, learning_rate=0.03, max_depth=8, random_state=42, n_jobs=-1, enable_categorical=True, tree_method='hist')
cat_temp = CatBoostRegressor(iterations=800, learning_rate=0.04, depth=8, random_seed=42, cat_features=categorical_cols, verbose=0)

print("Training temporary base models to gauge accuracy...")
lgb_temp.fit(X_train, y_train)
xgb_temp.fit(X_train, y_train)
cat_temp.fit(X_train, y_train)

val_pred_lgb = lgb_temp.predict(X_val)
val_pred_xgb = xgb_temp.predict(X_val)
val_pred_cat = cat_temp.predict(X_val)

print("Running Gradient Descent to lock in optimal weights...")
w1, w2, w3, b = 0.33, 0.33, 0.33, 0.0  
lr = 0.1
n = len(y_val)
for i in range(1500):
    y_hat = (w1 * val_pred_lgb) + (w2 * val_pred_xgb) + (w3 * val_pred_cat) + b
    error = y_hat - y_val
    w1 -= lr * ((2/n) * np.sum(error * val_pred_lgb))
    w2 -= lr * ((2/n) * np.sum(error * val_pred_xgb))
    w3 -= lr * ((2/n) * np.sum(error * val_pred_cat))
    b  -= lr * ((2/n) * np.sum(error))
    
    if i == 1499:
        print(f"Final Phase 1 MSE: {mean_squared_error(y_val, y_hat):.6f}")

print(f"Weights Locked In -> LGBM: {w1:.3f} | XGB: {w2:.3f} | CAT: {w3:.3f} | Bias: {b:.3f}")


print("\n--- PHASE 2: THE 100% RETRAIN ---")
print("Retraining all models on 100% of the dataset so they learn every pattern...")
lgb_final = LGBMRegressor(n_estimators=1000, learning_rate=0.03, max_depth=8, num_leaves=63, random_state=42, n_jobs=-1)
lgb_final.fit(X_FULL, y_FULL)

xgb_final = XGBRegressor(n_estimators=800, learning_rate=0.03, max_depth=8, random_state=42, n_jobs=-1, enable_categorical=True, tree_method='hist')
xgb_final.fit(X_FULL, y_FULL)

cat_final = CatBoostRegressor(iterations=800, learning_rate=0.04, depth=8, random_seed=42, cat_features=categorical_cols, verbose=0)
cat_final.fit(X_FULL, y_FULL)


print("\n--- PHASE 3: FINAL SUBMISSION ---")
# Predict the blind test set using the massive 100% trained models
print("Predicting unseen test data...")
test_pred_lgb = lgb_final.predict(X_test)
test_pred_xgb = xgb_final.predict(X_test)
test_pred_cat = cat_final.predict(X_test)

# Apply the mathematically honest weights we found in Phase 1!
final_predictions = (w1 * test_pred_lgb) + (w2 * test_pred_xgb) + (w3 * test_pred_cat) + b

# ---> THE NEGATIVE VALUE FIX <---
# We force any mathematically impossible negative predictions back to a realistic floor of 0
print("Applying Negative Value Fix (Clipping to 0)...")
final_predictions = np.clip(final_predictions, 0, None)

submission = pd.DataFrame({
    'Index': test_clean['Index'],
    'demand': final_predictions
})

submission_filename = 'prediction.csv'
submission.to_csv(submission_filename, index=False)
print(f"\ncompleted!!")

1. Loading datasets...
2. Engineering proven features...
3. Applying Advanced Bayesian Smoothed Target Encoding...

--- PHASE 1: FINDING HONEST WEIGHTS (80/20 SPLIT) ---
Training temporary base models to gauge accuracy...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004721 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15218
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 17
[LightGBM] [Info] Start training from score 0.093784
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Running Gradient Descent to lock in optimal weights...
Final Phase 1 MSE: 0.001096
Weights Locked In -> LGBM: 0.347 | XGB: 0.337 